# 02. Vision Tools — YOLO / SAM / DepthAnything을 LangGraph Tool로 래핑

## 학습 목표
1. 실제 CV 모델(YOLO, SAM, DepthAnything)을 LangGraph Tool로 래핑
2. Tool call → 결과 파싱 → 다음 에이전트로 전달 흐름 구현
3. 이미지/영상 입력 처리 파이프라인 구축
4. Claude Vision API + CV 툴 연동

## 구조
```
이미지 입력
    ↓
Vision Agent (Claude)
    ↓ tool_call
┌───────────────────────┐
│ YOLOTool  : 객체 탐지 │
│ SAMTool   : 세그먼트  │
│ DepthTool : 깊이 추정 │
└───────────────────────┘
    ↓ observation
Vision Agent (결과 요약)
    ↓
구조화된 Vision Report
```

In [ ]:
# ── 한글 폰트 & 환경 설정
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import os
from dotenv import load_dotenv
load_dotenv('../.env')

print('ANTHROPIC_API_KEY 로드:', 'OK' if os.getenv('ANTHROPIC_API_KEY') else 'MISSING')

In [ ]:
# 패키지 설치 (최초 1회)
# CV 모델 (USE_REAL_MODELS=True 시 필요)
# !pip install ultralytics segment-anything transformers

# 기본 패키지
!pip install langchain-ollama langchain langchain-core langgraph python-dotenv pillow opencv-python


---
## Part 1. 테스트 이미지 준비

CV 모델은 GPU가 없으면 느리므로, 두 가지 모드로 운영합니다:
- **Mock 모드**: 실제 모델 없이 구조/인터페이스 학습
- **Real 모드**: 실제 모델 로드 (GPU 권장)

아래 셀에서 `USE_REAL_MODELS = True`로 바꾸면 실제 모델이 실행됩니다.

In [ ]:
# ── 모드 설정
USE_REAL_MODELS = False  # True: 실제 CV 모델, False: Mock (빠른 학습용)

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import io, base64, json

# ── 테스트 이미지 생성 (실제 이미지가 없을 경우 사용)
def create_test_image(width=640, height=480, label='테스트 이미지') -> Image.Image:
    """간단한 테스트용 이미지 생성"""
    img_array = np.zeros((height, width, 3), dtype=np.uint8)
    # 배경: 하늘색
    img_array[:height//2, :] = [135, 206, 235]
    # 도로: 회색
    img_array[height//2:, :] = [128, 128, 128]
    # 차량 박스 (임의)
    img_array[200:320, 100:280] = [0, 0, 200]   # 파란 차
    img_array[200:320, 360:540] = [200, 0, 0]   # 빨간 차
    # 보행자
    img_array[180:350, 310:340] = [255, 200, 150]  # 사람
    return Image.fromarray(img_array)

test_image = create_test_image()

plt.figure(figsize=(8, 6))
plt.imshow(test_image)
plt.title('테스트 입력 이미지 (교차로 시뮬레이션)')
plt.axis('off')
plt.tight_layout()
plt.show()
print(f'이미지 크기: {test_image.size}')

In [ ]:
# ── 이미지 → base64 인코딩 (Claude Vision API용)
def image_to_base64(img: Image.Image, format='JPEG') -> str:
    """PIL Image를 base64 문자열로 변환 (Claude API에 전달용)"""
    buffer = io.BytesIO()
    img.save(buffer, format=format)
    return base64.b64encode(buffer.getvalue()).decode('utf-8')

def base64_to_image(b64_str: str) -> Image.Image:
    """base64 문자열을 PIL Image로 복원"""
    img_bytes = base64.b64decode(b64_str)
    return Image.open(io.BytesIO(img_bytes))

test_b64 = image_to_base64(test_image)
print(f'Base64 인코딩 완료: {len(test_b64)} 문자')
print(f'앞 50자: {test_b64[:50]}...')

---
## Part 2. CV 모델 래퍼 구현

각 CV 모델을 **동일한 인터페이스**로 래핑합니다:
- 입력: `image_b64` (base64 이미지) + 선택적 파라미터
- 출력: JSON 직렬화 가능한 딕셔너리

이렇게 하면 LangGraph Tool로 쉽게 등록할 수 있습니다.

In [ ]:
# ── YOLO 래퍼 (객체 탐지)

class YOLOWrapper:
    """YOLOv8 래퍼. USE_REAL_MODELS=True면 실제 모델 로드"""
    
    def __init__(self):
        self.model = None
        if USE_REAL_MODELS:
            try:
                from ultralytics import YOLO
                self.model = YOLO('yolov8n.pt')  # nano 모델 (가장 빠름)
                print('YOLOv8 모델 로드 완료')
            except Exception as e:
                print(f'YOLO 로드 실패 (Mock 모드로 전환): {e}')
        else:
            print('YOLO: Mock 모드')
    
    def detect(self, image_b64: str, conf_threshold: float = 0.5) -> dict:
        """
        객체 탐지 실행
        Returns: {'detections': [{'class': str, 'confidence': float, 'bbox': [x1,y1,x2,y2]}], ...}
        """
        if self.model is not None:
            # 실제 YOLO 실행
            img = base64_to_image(image_b64)
            results = self.model(img, conf=conf_threshold)
            detections = []
            for r in results:
                for box in r.boxes:
                    detections.append({
                        'class': r.names[int(box.cls)],
                        'confidence': float(box.conf),
                        'bbox': box.xyxy[0].tolist()
                    })
            return {
                'detections': detections,
                'count': len(detections),
                'model': 'yolov8n'
            }
        else:
            # Mock 결과 (구조 동일)
            return {
                'detections': [
                    {'class': 'car',    'confidence': 0.92, 'bbox': [100, 200, 280, 320]},
                    {'class': 'car',    'confidence': 0.88, 'bbox': [360, 200, 540, 320]},
                    {'class': 'person', 'confidence': 0.85, 'bbox': [310, 180, 340, 350]},
                    {'class': 'traffic light', 'confidence': 0.79, 'bbox': [280, 50, 310, 120]}
                ],
                'count': 4,
                'model': 'yolov8n-mock'
            }

yolo = YOLOWrapper()

# 테스트
yolo_result = yolo.detect(test_b64)
print('YOLO 탐지 결과:')
print(json.dumps(yolo_result, indent=2, ensure_ascii=False))

In [ ]:
# ── YOLO 탐지 결과 시각화
import matplotlib.patches as patches

def visualize_detections(image: Image.Image, detections: list, title='YOLO 탐지 결과'):
    fig, ax = plt.subplots(1, 1, figsize=(10, 7))
    ax.imshow(image)
    
    colors = {'car': 'blue', 'person': 'red', 'traffic light': 'green', 'default': 'yellow'}
    
    for det in detections:
        x1, y1, x2, y2 = det['bbox']
        color = colors.get(det['class'], colors['default'])
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-5, f"{det['class']} {det['confidence']:.2f}",
                color=color, fontsize=9, fontweight='bold')
    
    ax.set_title(title)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

visualize_detections(test_image, yolo_result['detections'])

In [ ]:
# ── SAM 래퍼 (세그멘테이션)

class SAMWrapper:
    """Segment Anything Model 래퍼"""
    
    def __init__(self):
        self.predictor = None
        if USE_REAL_MODELS:
            try:
                from segment_anything import sam_model_registry, SamPredictor
                sam = sam_model_registry['vit_b'](checkpoint='sam_vit_b.pth')
                self.predictor = SamPredictor(sam)
                print('SAM 모델 로드 완료')
            except Exception as e:
                print(f'SAM 로드 실패 (Mock 모드로 전환): {e}')
        else:
            print('SAM: Mock 모드')
    
    def segment(self, image_b64: str, bboxes: list = None) -> dict:
        """
        세그멘테이션 실행
        bboxes: YOLO 탐지 결과의 bbox 리스트 (프롬프트로 사용)
        Returns: {'segments': [{'class': str, 'area': int, 'mask_summary': str}], ...}
        """
        if self.predictor is not None:
            # 실제 SAM 실행 (YOLO bbox를 프롬프트로)
            img = np.array(base64_to_image(image_b64))
            self.predictor.set_image(img)
            segments = []
            for i, bbox in enumerate(bboxes or []):
                masks, scores, _ = self.predictor.predict(
                    box=np.array(bbox),
                    multimask_output=False
                )
                area = int(masks[0].sum())
                segments.append({
                    'id': i,
                    'area': area,
                    'score': float(scores[0]),
                    'mask_summary': f'{area}px 세그멘트 (신뢰도 {scores[0]:.2f})'
                })
            return {'segments': segments, 'model': 'sam-vit-b'}
        else:
            # Mock 결과
            mock_segments = []
            for i, bbox in enumerate(bboxes or [[100,200,280,320], [360,200,540,320]]):
                area = int((bbox[2]-bbox[0]) * (bbox[3]-bbox[1]) * 0.85)
                mock_segments.append({
                    'id': i,
                    'area': area,
                    'score': round(0.90 - i*0.03, 2),
                    'mask_summary': f'{area}px 세그멘트 (신뢰도 {0.90-i*0.03:.2f})'
                })
            return {'segments': mock_segments, 'model': 'sam-mock'}

sam = SAMWrapper()

# YOLO bbox로 SAM 실행
bboxes = [d['bbox'] for d in yolo_result['detections']]
sam_result = sam.segment(test_b64, bboxes=bboxes)
print('SAM 세그멘테이션 결과:')
print(json.dumps(sam_result, indent=2, ensure_ascii=False))

In [ ]:
# -- DepthAnything 래퍼 (깊이 추정)

class DepthWrapper:
    """DepthAnything v2 래퍼 (단안 깊이 추정)"""
    
    def __init__(self):
        self.pipe = None
        if USE_REAL_MODELS:
            try:
                from transformers import pipeline
                self.pipe = pipeline(
                    task="depth-estimation",
                    model="LiheYoung/depth-anything-small-hf"
                )
                print("DepthAnything 모델 로드 완료")
            except Exception as e:
                print(f"DepthAnything 로드 실패 (Mock 모드로 전환): {e}")
        else:
            print("DepthAnything: Mock 모드")

    def _get_bbox(self, det: dict):
        """다양한 키 이름에서 bbox 추출 (LLM이 다른 키를 쓸 경우 대비)"""
        for key in ["bbox", "bounding_box", "box", "coordinates"]:
            if key in det:
                return det[key]
        # bbox가 없으면 기본값 (이미지 중앙)
        return [160, 120, 480, 360]

    def estimate(self, image_b64: str, detections: list = None) -> dict:
        h = 480
        dets = detections or []
        if self.pipe is not None:
            img = base64_to_image(image_b64)
            depth_output = self.pipe(img)
            depth_array = np.array(depth_output["depth"])
            object_distances = []
            for det in dets:
                x1, y1, x2, y2 = [int(v) for v in self._get_bbox(det)]
                roi = depth_array[y1:y2, x1:x2]
                avg_depth = float(roi.mean()) if roi.size > 0 else 0
                object_distances.append({
                    "class": det.get("class", "unknown"),
                    "relative_depth": round(avg_depth, 3),
                    "distance_label": "가까움" if avg_depth > 0.7 else ("중간" if avg_depth > 0.4 else "멀음")
                })
            depth_norm = ((depth_array - depth_array.min()) /
                          (depth_array.max() - depth_array.min() + 1e-8) * 255).astype(np.uint8)
            depth_b64 = image_to_base64(Image.fromarray(depth_norm))
            return {"depth_map_b64": depth_b64, "object_distances": object_distances, "model": "depth-anything-small"}
        else:
            # Mock: bbox 하단 위치 기반 깊이 추정
            object_distances = []
            for det in dets:
                bbox = self._get_bbox(det)
                center_y = (bbox[1] + bbox[3]) / 2
                rel_depth = center_y / h
                label = "가까움" if rel_depth > 0.6 else ("중간" if rel_depth > 0.4 else "멀음")
                object_distances.append({
                    "class": det.get("class", "unknown"),
                    "relative_depth": round(rel_depth, 3),
                    "distance_label": label
                })
            return {"depth_map_b64": "mock_depth_map", "object_distances": object_distances, "model": "depth-anything-mock"}

depth = DepthWrapper()

depth_result = depth.estimate(test_b64, detections=yolo_result["detections"])
print("DepthAnything 결과:")
print(json.dumps({k: v for k, v in depth_result.items() if k != "depth_map_b64"},
                 indent=2, ensure_ascii=False))


---
## Part 3. LangGraph Tool 등록

CV 래퍼를 LangGraph/LangChain Tool로 래핑합니다.  
Claude API의 `tool_use` 형식에 맞게 스키마를 자동 생성합니다.

In [ ]:
# -- Pydantic 스키마 정의 (tool input validation)
from pydantic import BaseModel, Field, field_validator
from langchain_core.tools import tool
from typing import Optional, List, Any

class YOLOInput(BaseModel):
    image_b64: str = Field(description="base64 인코딩된 이미지 문자열")
    conf_threshold: float = Field(default=0.5, description="탐지 신뢰도 임계값 (0~1)")

class SAMInput(BaseModel):
    image_b64: str = Field(description="base64 인코딩된 이미지 문자열")
    bboxes: List[List[float]] = Field(default=[], description="YOLO bbox 목록")

    @field_validator("bboxes", mode="before")
    @classmethod
    def coerce_bboxes(cls, v):
        # dict로 넘어와도 빈 리스트로 처리
        if isinstance(v, dict):
            return []
        return v if v is not None else []

class DepthInput(BaseModel):
    image_b64: str = Field(description="base64 인코딩된 이미지 문자열")
    detections: List[dict] = Field(default=[], description="YOLO 탐지 결과 리스트")

    @field_validator("detections", mode="before")
    @classmethod
    def coerce_detections(cls, v):
        # LLM이 dict로 잘못 넘길 경우 자동 변환
        if isinstance(v, dict):
            result = []
            for cls_name, bboxes in v.items():
                if isinstance(bboxes, list):
                    for bbox in bboxes:
                        result.append({"class": cls_name, "bbox": bbox})
            return result
        return v if v is not None else []

@tool(args_schema=YOLOInput)
def yolo_detect(image_b64: str, conf_threshold: float = 0.5) -> str:
    """YOLOv8로 이미지에서 객체를 탐지합니다."""
    result = yolo.detect(image_b64, conf_threshold)
    return json.dumps(result, ensure_ascii=False)

@tool(args_schema=SAMInput)
def sam_segment(image_b64: str, bboxes: List[List[float]] = []) -> str:
    """Segment Anything Model로 이미지 객체의 윤곽을 추출합니다."""
    result = sam.segment(image_b64, bboxes=bboxes)
    return json.dumps(result, ensure_ascii=False)

@tool(args_schema=DepthInput)
def depth_estimate(image_b64: str, detections: List[dict] = []) -> str:
    """DepthAnything으로 이미지의 깊이를 추정합니다."""
    result = depth.estimate(image_b64, detections=detections)
    result_summary = {k: v for k, v in result.items() if k != "depth_map_b64"}
    result_summary["depth_map_available"] = result.get("depth_map_b64") != "mock_depth_map"
    return json.dumps(result_summary, ensure_ascii=False)

VISION_TOOLS = [yolo_detect, sam_segment, depth_estimate]
VISION_TOOL_MAP = {t.name: t for t in VISION_TOOLS}

print("Vision Tools 등록 완료:")
for t in VISION_TOOLS:
    print(f"  - {t.name}: {t.description[:60]}...")


In [ ]:
# ── Ollama LLM 설정
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage

client_llm = ChatOllama(model="llama3.2", temperature=0)

def call_llm(messages: list, system: str = "") -> str:
    lc_messages = []
    if system:
        lc_messages.append(SystemMessage(content=system))
    for m in messages:
        if isinstance(m, dict):
            content = m["content"] if isinstance(m["content"], str) else str(m["content"])
            lc_messages.append(
                HumanMessage(content=content) if m["role"] == "user" else AIMessage(content=content)
            )
        else:
            lc_messages.append(m)
    return client_llm.invoke(lc_messages).content

# Vision tool 바인딩
llm_with_vision_tools = client_llm.bind_tools(VISION_TOOLS)
print("Ollama Vision Agent 준비 완료")
print("모델:", client_llm.model)


---
## Part 4. Vision Agent 구현 (LangGraph)

Claude가 주도적으로 CV 툴을 선택하고, 결과를 통합하여 분석 리포트를 생성합니다.

In [ ]:
# ── Vision Agent State
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List
import operator

class VisionAgentState(TypedDict):
    messages: Annotated[List[dict], operator.add]  # 대화 이력
    image_b64: str                                  # 입력 이미지
    yolo_result: dict                               # YOLO 탐지 결과
    sam_result: dict                                # SAM 세그멘트 결과
    depth_result: dict                              # 깊이 추정 결과
    vision_report: str                              # 최종 분석 리포트
    step_count: int                                 # 반복 카운터

print('VisionAgentState 정의 완료')

In [ ]:
# ── Vision Agent 노드 (Ollama)

VISION_SYSTEM_PROMPT = """당신은 컴퓨터 비전 분석 전문가입니다.
제공된 CV 툴을 사용하여 이미지를 단계적으로 분석하세요.

분석 순서:
1. yolo_detect: 객체 탐지
2. sam_segment: 세그멘테이션
3. depth_estimate: 깊이 추정
4. 결과 통합 → 최종 분석 리포트 작성
"""

def vision_agent_node(state: VisionAgentState) -> VisionAgentState:
    step = state.get("step_count", 0)
    lc_messages = [SystemMessage(content=VISION_SYSTEM_PROMPT)]
    for m in state["messages"]:
        if isinstance(m, (HumanMessage, AIMessage, ToolMessage, SystemMessage)):
            lc_messages.append(m)
        elif isinstance(m, dict):
            content = m["content"] if isinstance(m["content"], str) else str(m["content"])
            lc_messages.append(
                HumanMessage(content=content) if m["role"] == "user" else AIMessage(content=content)
            )
    response = llm_with_vision_tools.invoke(lc_messages)
    tc_names = [tc["name"] for tc in response.tool_calls] if response.tool_calls else []
    print(f"  [Vision Agent step {step+1}] tool_calls: {tc_names if tc_names else '없음'}")
    return {"messages": [response], "step_count": step + 1}

print("Vision Agent 노드 정의 완료")


In [ ]:
# ── Vision Tool 실행 노드

def vision_tool_node(state: VisionAgentState) -> VisionAgentState:
    last_message = state["messages"][-1]
    image_b64 = state["image_b64"]
    tool_results = []
    new_state = {}

    for tool_call in last_message.tool_calls:
        tool_name    = tool_call["name"]
        tool_input   = tool_call["args"]
        tool_call_id = tool_call["id"]

        tool_input_with_image = {"image_b64": image_b64, **tool_input}
        print(f"  -> 툴 실행: {tool_name}")

        if tool_name in VISION_TOOL_MAP:
            result_str  = VISION_TOOL_MAP[tool_name].invoke(tool_input_with_image)
            result_dict = json.loads(result_str)

            if tool_name == "yolo_detect":
                new_state["yolo_result"] = result_dict
                print(f"  <- 탐지: {result_dict['count']}개 객체")
            elif tool_name == "sam_segment":
                new_state["sam_result"] = result_dict
                print(f"  <- 세그멘테이션: {len(result_dict['segments'])}개")
            elif tool_name == "depth_estimate":
                new_state["depth_result"] = result_dict
                print(f"  <- 깊이 추정 완료")

            tool_results.append(ToolMessage(content=result_str, tool_call_id=tool_call_id))
        else:
            tool_results.append(ToolMessage(
                content=f"알 수 없는 툴: {tool_name}", tool_call_id=tool_call_id
            ))

    new_state["messages"] = tool_results
    return new_state

print("Vision Tool 노드 정의 완료")


In [ ]:
# ── 조건부 엣지 & 그래프 구성

def vision_should_continue(state: VisionAgentState) -> str:
    if state.get("step_count", 0) >= 8:
        return END
    last_msg = state["messages"][-1]
    if getattr(last_msg, "tool_calls", None):
        return "vision_tool_node"
    return END

vision_workflow = StateGraph(VisionAgentState)
vision_workflow.add_node("vision_agent", vision_agent_node)
vision_workflow.add_node("vision_tool_node", vision_tool_node)

vision_workflow.set_entry_point("vision_agent")
vision_workflow.add_conditional_edges(
    "vision_agent",
    vision_should_continue,
    {"vision_tool_node": "vision_tool_node", END: END}
)
vision_workflow.add_edge("vision_tool_node", "vision_agent")

vision_app = vision_workflow.compile()
print("Vision Agent 그래프 컴파일 완료")


In [ ]:
# ── Vision Agent 실행

vision_initial_state = {
    "messages": [HumanMessage(content="교차로 이미지를 분석해주세요. YOLO로 객체를 탐지하고, SAM으로 세그멘테이션하고, 깊이 추정까지 완료한 뒤 안전 위험 요소 리포트를 작성해주세요.")],
    "image_b64": test_b64,
    "yolo_result": {},
    "sam_result": {},
    "depth_result": {},
    "vision_report": "",
    "step_count": 0
}

print("=== Vision Agent 실행 시작 ===")
print(f"태스크: {vision_initial_state['messages'][0].content}")
print()

final_vision_state = vision_app.invoke(vision_initial_state)

print()
print("=" * 60)
print("Vision Agent 최종 결과")
print("=" * 60)
last_msg = final_vision_state["messages"][-1]
print(last_msg.content if hasattr(last_msg, "content") else str(last_msg))
print(f"\n총 스텝: {final_vision_state['step_count']}")


---
## Part 5. Claude Vision API — 실제 이미지 이해

CV 툴 결과 + Claude의 멀티모달 이해를 결합합니다.  
Claude가 **이미지를 직접 보면서** CV 툴 결과를 해석합니다.

In [ ]:
# ── Ollama Vision 분석 (CV 결과 + LLM 통합 해석)
# 참고: Ollama 로컬 모델은 이미지를 직접 보지 못하므로
#       CV 툴(YOLO/SAM/Depth) 결과를 텍스트로 변환해서 전달

def analyze_with_ollama(cv_results: dict, question: str) -> str:
    """CV 툴 결과를 텍스트로 변환 → Ollama로 통합 분석"""
    cv_summary = json.dumps(cv_results, indent=2, ensure_ascii=False)
    prompt = f"""다음 CV 자동 분석 결과를 보고 질문에 답해주세요.

## CV 분석 결과
{cv_summary}

## 질문
{question}
"""
    return call_llm([{"role": "user", "content": prompt}],
                    system="당신은 컴퓨터 비전과 도로 안전 전문가입니다.")

# 통합 분석 실행
cv_results = {
    "yolo": yolo_result,
    "sam":  sam_result,
    "depth": {k: v for k, v in depth_result.items() if k != "depth_map_b64"}
}

print("=== Ollama + CV 툴 통합 분석 ===")
analysis = analyze_with_ollama(
    cv_results,
    "이 교차로의 안전 위험 요소를 구조화된 형식으로 분석해주세요."
)
print(analysis)


---
## Part 6. 영상(Video) 프레임 처리 파이프라인

영상을 프레임으로 분할 → 각 프레임 분석 → 시간축 이상 탐지

In [ ]:
# ── 영상 → 프레임 추출 유틸리티

def extract_frames(video_path: str = None, num_frames: int = 5) -> list:
    """
    영상에서 프레임을 균일하게 추출
    video_path=None이면 Mock 프레임 생성
    Returns: [PIL.Image, ...]
    """
    if video_path and os.path.exists(video_path):
        import cv2
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        indices = np.linspace(0, total_frames-1, num_frames, dtype=int)
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(Image.fromarray(frame_rgb))
        cap.release()
        return frames
    else:
        # Mock: 시간에 따라 약간씩 다른 프레임 생성
        frames = []
        for i in range(num_frames):
            img_array = np.zeros((480, 640, 3), dtype=np.uint8)
            img_array[:240, :] = [135, 206, 235]  # 하늘
            img_array[240:, :] = [128, 128, 128]   # 도로
            # 차량 위치가 프레임마다 이동
            x_offset = i * 40
            img_array[200:320, 100+x_offset:280+x_offset] = [0, 0, 200]
            frames.append(Image.fromarray(img_array))
        return frames

frames = extract_frames(num_frames=5)
print(f'{len(frames)}개 프레임 추출 완료')

# 프레임 시각화
fig, axes = plt.subplots(1, len(frames), figsize=(15, 3))
for i, (ax, frame) in enumerate(zip(axes, frames)):
    ax.imshow(frame)
    ax.set_title(f'Frame {i+1}')
    ax.axis('off')
plt.suptitle('추출된 프레임 (시간순)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 프레임 배치 분석 + 이상 탐지

def analyze_video_frames(frames: list, sample_interval: int = 1) -> dict:
    """
    영상 프레임 배치 분석
    Returns: {'frame_results': [...], 'anomalies': [...], 'summary': str}
    """
    frame_results = []
    
    for i, frame in enumerate(frames[::sample_interval]):
        frame_b64 = image_to_base64(frame)
        yolo_res = yolo.detect(frame_b64)
        
        frame_results.append({
            'frame_idx': i * sample_interval,
            'detections': yolo_res['detections'],
            'object_count': yolo_res['count'],
            'classes': list(set(d['class'] for d in yolo_res['detections']))
        })
        print(f'  Frame {i+1}/{len(frames)}: {yolo_res["count"]}개 객체 탐지')
    
    # 이상 탐지: 객체 수가 갑자기 변화하는 프레임
    counts = [fr['object_count'] for fr in frame_results]
    mean_count = np.mean(counts)
    std_count = np.std(counts)
    
    anomalies = [
        {
            'frame_idx': fr['frame_idx'],
            'count': fr['object_count'],
            'reason': f'객체 수 이상 ({fr["object_count"]} vs 평균 {mean_count:.1f})'
        }
        for fr in frame_results
        if abs(fr['object_count'] - mean_count) > std_count * 1.5
    ]
    
    return {
        'frame_results': frame_results,
        'anomalies': anomalies,
        'statistics': {
            'total_frames': len(frame_results),
            'avg_objects': round(mean_count, 1),
            'max_objects': max(counts),
            'min_objects': min(counts)
        }
    }

print('=== 영상 프레임 분석 시작 ===')
video_analysis = analyze_video_frames(frames)

print()
print('분석 통계:')
print(json.dumps(video_analysis['statistics'], indent=2, ensure_ascii=False))
print(f'\n이상 감지: {len(video_analysis["anomalies"])}건')
for anomaly in video_analysis['anomalies']:
    print(f'  Frame {anomaly["frame_idx"]}: {anomaly["reason"]}')

In [ ]:
# ── 영상 분석 결과 시각화

frame_counts = [fr['object_count'] for fr in video_analysis['frame_results']]
frame_indices = [fr['frame_idx'] for fr in video_analysis['frame_results']]

plt.figure(figsize=(10, 4))
plt.plot(frame_indices, frame_counts, 'b-o', linewidth=2, markersize=8, label='탐지 객체 수')
plt.axhline(y=video_analysis['statistics']['avg_objects'], color='r',
            linestyle='--', label=f'평균 ({video_analysis["statistics"]["avg_objects"]})')

for anomaly in video_analysis['anomalies']:
    plt.axvline(x=anomaly['frame_idx'], color='orange', alpha=0.5, linestyle=':', label='이상 탐지')

plt.xlabel('프레임 번호')
plt.ylabel('탐지 객체 수')
plt.title('영상 프레임별 객체 탐지 현황')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 정리

| 구성 요소 | 역할 | 인터페이스 |
|----------|------|----------|
| `YOLOWrapper` | 객체 탐지 | `detect(image_b64) → dict` |
| `SAMWrapper` | 세그멘테이션 | `segment(image_b64, bboxes) → dict` |
| `DepthWrapper` | 깊이 추정 | `estimate(image_b64, detections) → dict` |
| `yolo_detect` (Tool) | LangGraph 등록 | `@tool + Pydantic 스키마` |
| `VisionAgentState` | 결과 저장 | 각 툴 결과 필드 분리 |
| `vision_tool_node` | image_b64 자동 주입 | state에서 꺼내서 주입 |
| Claude Vision API | 이미지 직접 이해 | `content[{type:image}]` |
| `analyze_video_frames` | 영상 배치 처리 | 프레임 추출 → 순차 분석 |

## 다음 노트북
**03_rag_pipeline.ipynb**: ChromaDB 벡터 스토어 + RAG Agent 구현